In [ ]:
import argparse
import csv
import os
import numpy as np
import torch
import torch.nn as nn
from PIL import Image, ImageFile
from torchvision.transforms import v2
from torchvision.models import resnet18, ResNet18_Weights
from tqdm import tqdm
ImageFile.LOAD_TRUNCATED_IMAGES = True
NUM_CLASSES = 5
IMG_SIZE = 128
def load_data_to_ram(csv_path, images_dir, limit=None):
    if not os.path.exists(csv_path):
        return None, None
    with open(csv_path, newline="") as f:
        rows = list(csv.DictReader(f))
    if limit:
        rows = rows[:limit]
    num_samples = len(rows)
    X = np.zeros((num_samples, 3, IMG_SIZE, IMG_SIZE), dtype=np.float32)
    ids_or_labels = []
    opened_images = {}
    print(f"Загрузка и нарезка {num_samples}")
    for i, r in enumerate(tqdm(rows, desc=" RAM-Сборка")):
        img_id = r["image_id"]
        if img_id not in opened_images:
            opened_images[img_id] = Image.open(os.path.join(images_dir, img_id)).convert("RGB")
        im = opened_images[img_id]
        x, y, w, h = int(r["x"]), int(r["y"]), int(r["w"]), int(r["h"])
        crop = im.crop((x, y, x + w, y + h)).resize((IMG_SIZE, IMG_SIZE), resample=Image.Resampling.BILINEAR)
        arr = np.array(crop, dtype=np.float32).transpose(2, 0, 1) / 255.0
        X[i] = arr
        target = int(r["class"]) if "class" in r else int(r["window_id"])
        ids_or_labels.append(target)
    return torch.from_numpy(X), torch.tensor(ids_or_labels)
def build_model(device):
    model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
    model.fc = nn.Sequential(
        nn.Dropout(p=0.2),
        nn.Linear(model.fc.in_features, NUM_CLASSES)
    )
    return model.to(device, memory_format=torch.channels_last)
def main(argv=None):
    p = argparse.ArgumentParser()
    p.add_argument("--images-dir", required=True)
    p.add_argument("--data-dir", required=True)
    p.add_argument("--epochs", type=int, default=3)
    p.add_argument("--batch", type=int, default=512)
    p.add_argument("--lr", type=float, default=1e-3)
    p.add_argument("--out", default="submission_ram_speed.csv")
    a = p.parse_args(argv)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    X_train, Y_train = load_data_to_ram(os.path.join(a.data_dir, "windows_train_a.csv"), a.images_dir)
    X_test, test_wids = load_data_to_ram(os.path.join(a.data_dir, "windows_test_a.csv"), a.images_dir)

    gpu_train_transform = v2.Compose([
        v2.RandomHorizontalFlip(p=0.5),
        v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    gpu_test_transform = v2.Compose([
        v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    model = build_model(device)
    use_amp = (device == "cuda")
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    opt = torch.optim.AdamW(model.parameters(), lr=a.lr, weight_decay=1e-2)
    crit = nn.CrossEntropyLoss(label_smoothing=0.1)

    num_train = len(X_train)
    num_batches = int(np.ceil(num_train / a.batch))
    for ep in range(a.epochs):
        model.train()
        indices = torch.randperm(num_train)
        X_train = X_train[indices]
        Y_train = Y_train[indices]
        correct, total = 0, 0
        pbar = tqdm(range(num_batches), desc=f"Эпоха {ep+1}/{a.epochs}")
        for b in pbar:
            start_idx = b * a.batch
            end_idx = min(start_idx + a.batch, num_train)
            xb = X_train[start_idx:end_idx].to(device, non_blocking=True)
            yb = Y_train[start_idx:end_idx].to(device, non_blocking=True)
            xb = gpu_train_transform(xb).to(memory_format=torch.channels_last)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=use_amp):
                out = model(xb)
                loss = crit(out, yb)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            correct += (out.argmax(1) == yb).sum().item()
            total += len(yb)
            pbar.set_postfix({"acc": f"{correct/total:.3f}"})
    model.eval()
    num_test = len(X_test)
    num_test_batches = int(np.ceil(num_test / a.batch))
    all_preds = []
    with torch.inference_mode():
        for b in tqdm(range(num_test_batches), desc="Предикт"):
            start_idx = b * a.batch
            end_idx = min(start_idx + a.batch, num_test)
            xb = X_test[start_idx:end_idx].to(device, non_blocking=True)
            xb = gpu_test_transform(xb).to(memory_format=torch.channels_last)
            with torch.amp.autocast("cuda", enabled=use_amp):
                logits = model(xb)
            all_preds.extend(logits.argmax(dim=1).cpu().tolist())
    with open(a.out, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["window_id", "class"])
        for wid, pred in zip(test_wids.tolist(), all_preds):
            w.writerow([wid, pred])
    print(f"\nКод завершен")
    return 0
if __name__ == "__main__":
    main([
        "--images-dir", "/kaggle/input/datasets/ilyaselivanov1/dataset/umnaya-polka/images",
        "--data-dir", "/kaggle/input/datasets/ilyaselivanov1/dataset/umnaya-polka/track_a",
        "--epochs", "3",
        "--batch", "512"
    ])

Загрузка и нарезка 41180


 RAM-Сборка:  30%|███       | 12376/41180 [02:21<06:46, 70.84it/s] 